# SNN Dataset Preprocessing Tutorial
## Step-by-step guide to preprocessing spike train data

This notebook demonstrates various preprocessing techniques for SNN datasets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from snn_data_preprocessing import SNNDataPreprocessor

%matplotlib inline

## 1. Quick Start: Complete Pipeline

Run the entire preprocessing pipeline in one command:

In [ ]:
# Initialize preprocessor
preprocessor = SNNDataPreprocessor(random_state=42)

# Run complete pipeline
result = preprocessor.preprocess_pipeline(
    dataset_name='classification_rate',
    remove_noise=True,
    normalize=True,
    balance=False,
    augment=False,
    split=True,
    save_processed=True
)

## 2. Step-by-Step: Individual Operations

Let's go through each preprocessing step individually for better understanding.

In [ ]:
# Load dataset
preprocessor = SNNDataPreprocessor()
dataset = preprocessor.load_dataset('classification_rate')

spike_trains = dataset['spike_trains']
labels = dataset['labels']

print(f"Original shape: {spike_trains.shape}")
print(f"Labels shape: {labels.shape}")

### 2.1 Remove Noisy Neurons

In [ ]:
# Remove neurons with abnormal firing rates
filtered_trains, kept_neurons = preprocessor.remove_noisy_neurons(
    spike_trains, 
    threshold=0.01
)

print(f"\nAfter noise removal: {filtered_trains.shape}")
print(f"Kept neurons: {kept_neurons[:10]}...")  # First 10

### 2.2 Remove Silent Samples

In [ ]:
# Remove samples with too few spikes
cleaned_trains, cleaned_labels = preprocessor.remove_silent_samples(
    filtered_trains,
    labels,
    min_spikes=10
)

print(f"\nAfter removing silent samples: {cleaned_trains.shape}")

### 2.3 Normalize Firing Rates

In [ ]:
# Compare before and after normalization
print("Before normalization:")
print(f"  Mean firing rate: {cleaned_trains.mean():.4f}")
print(f"  Std firing rate: {cleaned_trains.std():.4f}")

# Normalize
normalized_trains = preprocessor.normalize_spike_rates(
    cleaned_trains,
    method='global'  # Try 'per_neuron' or 'per_sample'
)

print("\nAfter normalization:")
print(f"  Mean firing rate: {normalized_trains.mean():.4f}")
print(f"  Std firing rate: {normalized_trains.std():.4f}")

### 2.4 Balance Classes (Optional)

In [ ]:
# Check current class distribution
unique, counts = np.unique(cleaned_labels, return_counts=True)
print("Current distribution:")
for cls, count in zip(unique, counts):
    print(f"  Class {cls}: {count} samples")

# Balance if needed
if counts.max() - counts.min() > 50:  # If imbalanced
    balanced_trains, balanced_labels = preprocessor.balance_classes(
        normalized_trains,
        cleaned_labels,
        method='undersample'  # or 'oversample'
    )
else:
    print("\n✓ Classes are already balanced")
    balanced_trains = normalized_trains
    balanced_labels = cleaned_labels

### 2.5 Data Augmentation

In [ ]:
# Take one sample for demonstration
sample_idx = 0
original_sample = balanced_trains[sample_idx:sample_idx+1]

# Apply different augmentations
jittered = preprocessor.augment_jitter(original_sample, jitter_std=2.0)
dropout = preprocessor.augment_dropout(original_sample, dropout_rate=0.2)
noisy = preprocessor.augment_noise(original_sample, noise_rate=0.02)
warped = preprocessor.augment_time_warp(original_sample, warp_factor=0.1)

# Visualize
fig, axes = plt.subplots(5, 1, figsize=(14, 12))

augmentations = [
    (original_sample, 'Original'),
    (jittered, 'Temporal Jitter'),
    (dropout, 'Spike Dropout'),
    (noisy, 'Added Noise'),
    (warped, 'Time Warped')
]

for ax, (data, title) in zip(axes, augmentations):
    spike_times, neuron_ids = np.where(data[0].T)
    ax.scatter(spike_times, neuron_ids, s=3, c='black', marker='|')
    ax.set_ylabel('Neuron')
    ax.set_title(f'{title} (Total spikes: {data.sum():.0f})')
    ax.set_xlim(0, data.shape[2])

axes[-1].set_xlabel('Time Step')
plt.tight_layout()
plt.savefig('augmentation_examples.png', dpi=300)
plt.show()

### 2.6 Train/Val/Test Split

In [ ]:
# Split the data
splits = preprocessor.split_dataset(
    balanced_trains,
    balanced_labels,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    stratify=True
)

# Access splits
train_spikes = splits['train']['spike_trains']
train_labels = splits['train']['labels']

val_spikes = splits['val']['spike_trains']
val_labels = splits['val']['labels']

test_spikes = splits['test']['spike_trains']
test_labels = splits['test']['labels']

print("\n✓ Data ready for training!")

## 3. Data Quality Analysis

In [ ]:
# Comprehensive quality analysis
quality_stats = preprocessor.analyze_data_quality(
    train_spikes,
    train_labels,
    save_path='quality_analysis.png'
)

## 4. Compare Before and After Preprocessing

In [ ]:
# Load original data again
original_dataset = preprocessor.load_dataset('classification_rate')
original_spikes = original_dataset['spike_trains']

# Compare statistics
print("BEFORE PREPROCESSING:")
print(f"  Shape: {original_spikes.shape}")
print(f"  Total spikes: {original_spikes.sum():,.0f}")
print(f"  Sparsity: {(original_spikes.sum() / original_spikes.size) * 100:.2f}%")
print(f"  Mean spikes/sample: {original_spikes.sum(axis=(1,2)).mean():.2f}")

print("\nAFTER PREPROCESSING:")
print(f"  Shape: {train_spikes.shape}")
print(f"  Total spikes: {train_spikes.sum():,.0f}")
print(f"  Sparsity: {(train_spikes.sum() / train_spikes.size) * 100:.2f}%")
print(f"  Mean spikes/sample: {train_spikes.sum(axis=(1,2)).mean():.2f}")

## 5. Save Preprocessed Data

In [ ]:
# Save to disk
import os

output_dir = 'my_preprocessed_data'
os.makedirs(output_dir, exist_ok=True)

# Save splits
np.save(f'{output_dir}/train_spike_trains.npy', train_spikes)
np.save(f'{output_dir}/train_labels.npy', train_labels)
np.save(f'{output_dir}/val_spike_trains.npy', val_spikes)
np.save(f'{output_dir}/val_labels.npy', val_labels)
np.save(f'{output_dir}/test_spike_trains.npy', test_spikes)
np.save(f'{output_dir}/test_labels.npy', test_labels)

print(f"✓ Preprocessed data saved to: {output_dir}/")
print("\nTo load:")
print("  train_spikes = np.load('my_preprocessed_data/train_spike_trains.npy')")
print("  train_labels = np.load('my_preprocessed_data/train_labels.npy')")

## 6. Ready for Model Training!

Your preprocessed data is now ready to use with any SNN framework:

```python
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert to PyTorch
train_dataset = TensorDataset(
    torch.FloatTensor(train_spikes),
    torch.LongTensor(train_labels)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Now train your SNN model!
```